# SAE Feature Steering: Precise Interpretable Interventions

This notebook demonstrates IRTK's `sae_feature_steering` module:

1. **Feature steer** – amplify/suppress individual SAE features
2. **Multi-feature steer** – modify multiple features at once
3. **Find steering features** – discover features that drive a behavior
4. **Feature ablation effect** – measure impact of zeroing features
5. **Clamped generation** – generate with feature forced to a value

## Why This Matters

SAE feature steering enables fine-grained behavioral control by clamping or scaling individual SAE features. Unlike residual stream steering (which modifies a direction in a polysemantic space), feature-level interventions target specific interpretable features, enabling precise and predictable effects.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Turner et al. (2023) "Activation Addition"](https://arxiv.org/abs/2308.10248) — Steering model behavior by adding vectors to activations

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.sae import SparseAutoencoder
from irtk.sae_feature_steering import (
    feature_steer, multi_feature_steer,
    find_steering_features, feature_ablation_effect,
    clamped_feature_generation,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
sae = SparseAutoencoder(16, 32, key=jax.random.PRNGKey(0))
tokens = jnp.array([0, 1, 2, 3, 4, 5])
def metric_fn(logits): return float(logits[-1, 0])

In [ ]:
result = feature_steer(model, tokens, sae, "blocks.0.hook_resid_post", feature_idx=0, alpha=3.0)
print(f"Original feature activation: {result['feature_activation']:.4f}")
print(f"Max logit diff: {result['logit_diff']:.4f}")

In [ ]:
multi = multi_feature_steer(model, tokens, sae, "blocks.0.hook_resid_post",
                           feature_interventions={0: 2.0, 5: 0.0, 10: 3.0})
print(f"Features modified: {multi['n_features_modified']}")
print(f"Max logit diff: {multi['logit_diff']:.4f}")

In [ ]:
steering = find_steering_features(model, tokens, sae, "blocks.0.hook_resid_post", metric_fn, top_k=5)
print(f"Baseline metric: {steering['baseline_metric']:.4f}")
print(f"Top positive features: {steering['top_positive'][:3]}")
print(f"Top negative features: {steering['top_negative'][:3]}")

In [ ]:
ablation = feature_ablation_effect(model, tokens, sae, "blocks.0.hook_resid_post", metric_fn, features=[0, 1, 2, 3])
print(f"Ablation effects: {ablation['ablation_effects']}")
print(f"Most critical: feature {ablation['most_critical']}")

In [ ]:
gen = clamped_feature_generation(model, tokens, sae, "blocks.0.hook_resid_post",
                               feature_idx=0, clamp_value=2.0, max_new_tokens=5, temperature=0.5)
print(f"Generated {gen['n_generated']} tokens with feature {gen['clamped_feature']} clamped to {gen['clamp_value']}")
print(f"Token sequence: {gen['generated_tokens']}")